# i+1 Story SLM — QLoRA fine-tuning on Colab GPU

Trains **Qwen3-4B-Instruct** with 4-bit QLoRA on the i+1 comprehensible-input story dataset,
then runs the full base-vs-tuned eval (deterministic validators **+** LLM judge **+** cloze
inferability) on the golden set and the held-out set.

**Budget:** $10 = ~100 compute units. Develop on **L4** (~20 h), reserve **A100** for the one
final run. Disconnect the runtime the moment a cell finishes - units burn on connection time.
See `docs/COLAB_PLAN.md` for the full spend plan.

Everything the notebook calls already exists in the repo; the CLI flags are wired.


## Step 0 - pick the GPU runtime

**Runtime -> Change runtime type -> L4 GPU** (switch to A100 only for the final full run).
Then run the cell below to confirm the GPU is attached.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Step 1 - clone the repo and install

Installs the package with its `train` extras (torch/transformers/trl/peft/datasets/accelerate)
plus `bitsandbytes` for 4-bit QLoRA (CUDA only).


In [ ]:
REPO_URL = 'https://github.com/1624252/slm.git'  # <-- your GitHub repo

import os
if not os.path.isdir('slm'):
    !git clone $REPO_URL
%cd slm
!pip -q install -e '.[train]' bitsandbytes
print('\nInstalled.')

## Step 2 - API key from Colab Secrets (judge + tracking)

The LLM judge and LangSmith tracking read keys from the environment. **Do not paste keys into
the notebook.** Add them in Colab's **Secrets** panel (key icon, left sidebar), then this cell
loads them:

| Secret name | Value |
| --- | --- |
| `OPENAI_API_KEY` | your `tfy_va_...` key |
| `OPENAI_BASE_URL` | `https://tfy.promptlens.trilogy.com/v1` |
| `JUDGE_MODEL` | `claude-group/claude-sonnet-4-6` |
| `LANGSMITH_API_KEY` | your `lsv2_pt_...` key (optional) |

Toggle **Notebook access** on for each. If a secret is missing the eval still runs - it just
skips the judge (deterministic-only) or the LangSmith push.


In [ ]:
from google.colab import userdata
import os

# Copy Colab secrets -> environment (islm reads these; no .env needed on Colab).
_SECRETS = ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'JUDGE_MODEL',
            'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT']
for name in _SECRETS:
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
    except Exception:
        pass  # secret not set / access not granted -> feature degrades gracefully
os.environ.setdefault('LANGSMITH_PROJECT', 'islm-evals')

print('judge key set :', bool(os.environ.get('OPENAI_API_KEY')))
print('judge model   :', os.environ.get('JUDGE_MODEL', '(default)'))
print('langsmith set :', bool(os.environ.get('LANGSMITH_API_KEY')))

## Step 3 - build the dataset (reproducible from seeds)

`data/dataset_v1/` is git-ignored (large + regenerable), so we regenerate it here - fully
deterministic from the seeds, ~0.2 ms/story. Each story teaches **2-3 target words**, <=1 new
word/sentence, every target recurring >=3x.

`N_EN/N_ZH/N_JA` control size. Defaults are a **budget-sized training subset**; raise them (up
to 115000 / 15000 / 15000) to rebuild the full 144k corpus if you have units to spare.


In [ ]:
N_EN, N_ZH, N_JA = 20000, 5000, 5000  # budget subset; full = 115000/15000/15000

!python -m islm.datagen.synth --n $N_EN --language en --seed 42 --out data/generated/synth_en
!python -m islm.datagen.synth --n $N_ZH --language zh --seed 43 --out data/generated/synth_zh
!python -m islm.datagen.synth --n $N_JA --language ja --seed 44 --out data/generated/synth_ja

# Second pass: dedup + quality filter + revalidate (global -> no train/test leakage).
for L in ['en', 'zh', 'ja']:
    !python -m islm.datagen.curate --in data/generated/synth_$L --out data/curated/synth_$L

# Recombine into one multilingual dataset.
!mkdir -p data/dataset_v1
for S in ['train', 'val', 'test']:
    !cat data/curated/synth_en/$S.jsonl data/curated/synth_zh/$S.jsonl data/curated/synth_ja/$S.jsonl > data/dataset_v1/$S.jsonl
!wc -l data/dataset_v1/*.jsonl

## Step 4 - smoke test (Phase 0, ~3 units)

One tiny run to catch env bugs before the real burn: confirms QLoRA loads Qwen3-4B in 4-bit and
a train step + save works. `--smoke` uses tiny settings.


In [ ]:
BASE = 'Qwen/Qwen3-4B-Instruct'  # for a T4 (16 GB) fall back to 'Qwen/Qwen3-1.7B'

!python -m islm.train.sft --data data/curated/synth_en --base $BASE \
    --qlora --smoke --out outputs/colab_smoke
print('\nSmoke OK if an adapter was written to outputs/colab_smoke.')

## Step 5 - QLoRA fine-tune (aligned recipe)

Best-known recipe from the local runs, GPU-scaled: rank 32 / alpha 64, lr 2e-4, cosine schedule
with warmup + weight decay + grad clipping (wired inside `islm.train.sft`); `--merge` writes a
standalone fp16 model for eval/upload.

Trains on the **full multilingual** `data/dataset_v1`. For a cheap sweep first, point `--data`
at `data/curated/synth_en` with `--epochs 1`.


In [ ]:
!python -m islm.train.sft --data data/dataset_v1 --base $BASE --qlora \
    --epochs 3 --lr 2e-4 --lora-r 32 --lora-alpha 64 \
    --max-seq-len 1024 --grad-accum 8 --merge \
    --out outputs/qwen3_4b_qlora
print('\nDone. Adapter + merged model in outputs/qwen3_4b_qlora.')

## Step 6 - evaluate (base vs tuned, all criteria, tracked)

Runs the model on both eval targets. Each reports **every criterion** with base + tuned
columns: deterministic (hard-pass, OOV, <=1-new, recurrence), the 8-dim LLM judge rubric (incl.
coherence + interestingness), and cloze inferability. The judge fires automatically because
Step 2 set the key.

- **Golden eval** - the every-commit correctness gate (`evals/golden/golden.jsonl`).
- **Held-out eval** - wider coverage sweep across en/zh/ja + adversarial.


In [ ]:
# Golden set (all languages).
!python -m islm.eval.run --golden \
    --base-path $BASE --tuned-path $BASE --tuned-adapter outputs/qwen3_4b_qlora \
    --no-think --track --run-label colab-qwen3-4b-golden \
    --dataset data/dataset_v1 --epochs 3 --out evals/colab_golden

In [ ]:
# Held-out set (all languages) + adversarial.
!python -m islm.eval.run \
    --base-path $BASE --tuned-path $BASE --tuned-adapter outputs/qwen3_4b_qlora \
    --adversarial --no-think --track --run-label colab-qwen3-4b \
    --dataset data/dataset_v1 --epochs 3 --out evals/colab_heldout

In [ ]:
# Show the result tables inline.
import glob
for f in sorted(glob.glob('evals/colab_*/results*.md')):
    print('=== ' + f + ' ===')
    print(open(f, encoding='utf-8').read())

## Step 7 - download the adapter + results

Zips the trained adapter, merged model, eval outputs, and the updated `runs.jsonl` leaderboard
so you can pull them locally and record the run in `evals/RESULTS_LOG.md`.


In [ ]:
!zip -qr colab_artifacts.zip outputs/qwen3_4b_qlora evals/colab_golden evals/colab_heldout evals/results/runs.jsonl 2>/dev/null
from google.colab import files
files.download('colab_artifacts.zip')

## After Colab (local, free)

1. Unzip `colab_artifacts.zip` into the repo.
2. Record the run in `evals/RESULTS_LOG.md` (config + base->tuned deltas) and update
   `evals/LEADERBOARD.md`.
3. Commit (data stays git-ignored; only results/adapter-summaries are tracked).

**Unit discipline:** disconnect the runtime now. Sweep small, run big once. If 4B is tight on
VRAM, drop `--max-seq-len` before dropping the model; last resort switch `BASE` to
`Qwen/Qwen3-1.7B`.
